In [ ]:
import os
import glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (15, 5)

DATA_DIR = os.environ.get("BRATS_DATA_DIR", "../data/BraTS-GLI/extracted_data/")

mask_files = glob.glob(os.path.join(DATA_DIR, "**", "*-seg.nii.gz"), recursive=True)

patient_paths = sorted([os.path.dirname(f) for f in mask_files])

print(f"Num patients {len(patient_paths)}")

patient_path = patient_paths[0]
sample_patient = os.path.basename(patient_path)

print(f"Patient EDA {sample_patient}")
print(f"Full path {patient_path}")

In [ ]:
modalities = ["t1n", "t1c", "t2w", "t2f", "seg"]
volumes = {}

print(f"{'Modality':<10} | {'Shape X, Y, Z':<18} | {'Min':<8} | {'Max':<10} | {'Data type'}")
print("-" * 70)

for mod in modalities:
    file_pattern = os.path.join(patient_path, f"*{mod}.nii.gz")
    file_path = glob.glob(file_pattern)[0]

    img = nib.load(file_path)
    data = img.get_fdata()
    volumes[mod] = data

    print(f"{mod.upper():<10} | {str(data.shape):<18} | {np.min(data):<8.1f} | {np.max(data):<10.1f} | {data.dtype}")

In [ ]:
mask = volumes["seg"]
unique_labels, counts = np.unique(mask, return_counts=True)


brats_labels_dict = {
    0: "Background",
    1: "Necrotic Tumor Core (NCR)",
    2: "Edema (ED)",
    3: "Enhancing Tumor (ET)",
    4: "Enhancing Tumor (old label convention)"
}

print("Segmentation mask label count")
total_voxels = mask.size

for label, count in zip(unique_labels, counts):
    label_name = brats_labels_dict.get(int(label), "Unknown label")
    percentage = (count / total_voxels) * 100

    if label == 0:
        print(f"Label {int(label)}: {label_name:<45} | {count:>10} voxels ({percentage:>5.2f}%)")
    else:
        volume_cm3 = count / 1000
        print(f"Label {int(label)}: {label_name:<45} | {count:>10} voxels ({percentage:>5.2f}%) | {volume_cm3:.2f} cm³")

In [ ]:
def plot_middle_slices(volume, title, cmap="gray"):
    coords = np.argwhere(volumes["seg"] > 0)
    if len(coords) > 0:
        x_mid, y_mid, z_mid = coords.mean(axis=0).astype(int)
    else:
        # no tumor -> fall back to volume center
        x_mid, y_mid, z_mid = [dim // 2 for dim in volume.shape]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    ax = axes[0]
    ax.imshow(np.rot90(volume[:, :, z_mid]), cmap=cmap)
    ax.set_title("Axial Axis Z")
    ax.axis("off")

    ax = axes[1]
    ax.imshow(np.rot90(volume[:, y_mid, :]), cmap=cmap)
    ax.set_title("Coronal Axis Y")
    ax.axis("off")

    ax = axes[2]
    ax.imshow(np.rot90(volume[x_mid, :, :]), cmap=cmap)
    ax.set_title("Sagittal Axis X")
    ax.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

plot_middle_slices(volumes["t1c"], "T1c modality with contrast")
plot_middle_slices(volumes["seg"], "Segmentation mask ground truth", cmap="tab10")

In [ ]:
plt.figure(figsize=(15, 6))

for mod in ["t1n", "t1c", "t2w", "t2f"]:
    data = volumes[mod]
    # only brain voxels, background is 0 after skull-stripping
    brain_voxels = data[data > 0]

    sns.kdeplot(brain_voxels, label=mod.upper(), fill=True, alpha=0.3)

plt.title("Voxel intensity plot for different modalities", fontsize=14)
plt.xlabel("Intensity (voxel value)")
plt.ylabel("Density")
plt.xlim(0, 3000)
plt.legend()
plt.show()

In [ ]:
from collections import Counter

parent_folders = [os.path.basename(os.path.dirname(p)) for p in patient_paths]
distribution = Counter(parent_folders)

print("training contains")
for folder_name, count in distribution.items():
    print(f"Folder {folder_name:<30} | {count:>5} patients count")

print("-" * 50)
print(f"overall   | {len(patient_paths):>5} patients")

1. Data size and shape

    - 1621 patients

    - Dimensions (182, 218, 182). Scans are cropped tightly, no wasted black background around the head.

2. Label analysis and a missing class

    - Patient BraTS-GLI-00005-100 is missing label 1 (necrotic core). Not every tumor has a necrotic component.

    - Background dominates the volume (99.62%).

3. Intensity histogram (KDE)

    - T2W and T2F peak mostly in the 400-600 range. T1N and T1C are much more spread out, peaking around 2000 - motivates per-volume normalization.

4. Mask and scan slice visualization

    - Skull-stripping applied, background is black (0.0).

    - The segmentation mask (bottom row) aligns geometrically 100% with the anatomy above it - the NIfTI affine transforms are consistent across modalities. Brown surrounds the edema, blue marks the tumor tissue.